# NPL Symptom Extraction Training
Loads processed TF-IDF features, trains a Keras classifier, evaluates, and converts to TFLite (INT8).

In [ ]:
import numpy as np
from pathlib import Path
from tensorflow import keras
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import classification_report, confusion_matrix

ROOT = Path('..')
DATA_DIR = ROOT / 'data' / 'processed' / 'text_classifier'
X_train = np.load(DATA_DIR / 'X_train.npy')
y_train = np.load(DATA_DIR / 'y_train.npy')
X_test = np.load(DATA_DIR / 'X_test.npy')
y_test = np.load(DATA_DIR / 'y_test.npy')
num_classes = len(set(y_train))
y_train_o = keras.utils.to_categorical(y_train, num_classes)
y_test_o = keras.utils.to_categorical(y_test, num_classes)

In [ ]:
model = keras.Sequential([
    keras.layers.Input(shape=(X_train.shape[1],)),
    keras.layers.Dense(256, activation='relu'),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dense(num_classes, activation='softmax'),
])
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
callbacks = [keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)]
history = model.fit(X_train, y_train_o, epochs=30, batch_size=64, validation_split=0.1, callbacks=callbacks)

In [ ]:
loss, acc = model.evaluate(X_test, y_test_o)
print('Test accuracy:', acc)
y_pred = model.predict(X_test).argmax(axis=1)
print(classification_report(y_test, y_pred))
import matplotlib.pyplot as plt
import seaborn as sns
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=False, cmap='Blues')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
# Convert to TFLite (default optimization, then INT8)
import tensorflow as tf
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
# Representative dataset for INT8 quantization - use a small subset
def representative_dataset():
    for i in range(min(100, X_train.shape[0])):
        yield [X_train[i:i+1].astype('float32')]
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.uint8
converter.inference_output_type = tf.uint8
tflite_model = converter.convert()
(Path('..') / 'models').mkdir(parents=True, exist_ok=True)
with open(Path('..') / 'models' / 'npl_symptom_v1.tflite', 'wb') as f:
    f.write(tflite_model)
print('Saved TFLite model to models/npl_symptom_v1.tflite')